In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

In [2]:
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

In [3]:
def carregar_dados_csv(nome_arquivo):

    caminho = RAW_DIR / nome_arquivo
    if caminho.exists():
        try:
            df = pd.read_csv(caminho)
            return df
        except Exception as e:
            print(f"Erro ao carregar o arquivo {nome_arquivo}: {e}")
            return None
    else:
        print(f"Arquivo {nome_arquivo} não encontrado no diretório {RAW_DIR}.")
        return None
    

In [ ]:
def relatorio_limpeza(df, nome_tabela):

    print(f"Relatorio de Limpeza de Dados para a tabela: {nome_tabela.upper()}")
    print(f"{'='*60}")
    print(f"Shape original: {df.shape}")
    print(f"Memoria utilizada: {df.memory_usage(deep=True).sum() / 1024 ** 2:.2f} MB")
    print(f"\n Valores Nulos:")
    nulos = df.isnull().sum()
    if nulos.sum() > 0:
        print(nulos[nulos > 0])
    else:
        print("Nenhum valor nulo encontrado.")
    print(f"Duplicatas: {df.duplicated().sum()}")
    print(f"Tipos de dados:")
    print(df.dtypes)
    return nulos

In [4]:
def salvar_dataset(df, nome_arquivo):
    caminho = PROCESSED_DIR / nome_arquivo
    df.to_csv(caminho, index=False)
    print(f"Dataset salvo em: {caminho}")
    return caminho

In [5]:
arquivos = {
    'orders': 'olist_orders_dataset.csv',
    'customers': 'olist_customers_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'orders_payments': 'olist_order_payments_dataset.csv',
    'orders_reviews': 'olist_order_reviews_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv'
}

dados = {}
for nome, arquivo in arquivos.items():
    df = carregar_dados_csv(arquivo)
    if df is not None:
        dados[nome] = df

In [ ]:
print("Limpando dados da tabela Orders")
print("="*60)

orders = dados['orders'].copy()

#Converter datas
colunas_data = ['order_purchase_timestamp', 
                'order_approved_at', 
                'order_delivered_carrier_date', 
                'order_delivered_customer_date', 
                'order_estimated_delivery_date']

for col in colunas_data:
    if col in orders.columns:
        orders[col] = pd.to_datetime(orders[col], errors='coerce')
        print(f"Coluna {col} convertida para datetime.")

# Tratando valores nulos

orders['order_approved_at'] = orders['order_approved_at'].fillna(orders['order_purchase_timestamp'])
orders['order_delivered_carrier_date'] = orders['order_delivered_carrier_date'].fillna(orders['order_estimated_delivery_date'])
orders['order_delivered_customer_date'] = orders['order_delivered_customer_date'].fillna(orders['order_estimated_delivery_date'])

#Remove pedidos com status indefinido

orders = orders[orders['order_status'].notna()]

#Remover duplicatas

orders = orders.drop_duplicates(subset=['order_id'])

#Criar colunas derivadas

orders['order_purchase_date'] = orders['order_purchase_timestamp'].dt.date
orders['order_purchase_year'] = orders['order_purchase_timestamp'].dt.year
orders['order_purchase_month'] = orders['order_purchase_timestamp'].dt.month
orders['order_purchase_day'] = orders['order_purchase_timestamp'].dt.day
orders['order_purchase_weekday'] = orders['order_purchase_timestamp'].dt.day_name()
orders['order_purchase_hour'] = orders['order_purchase_timestamp'].dt.hour

#Calcular tempo de entrega em dias

if 'order_delivered_customer_date' in orders.columns and 'order_purchase_timestamp' in orders.columns:
    orders['delivery_time_days'] = (
        orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']
        ).dt.days

#Salvar dataset limpo

salvar_dataset(orders, 'orders_cleaned.csv')
dados['orders'] = orders

In [ ]:
#Limpeza de dados da tabela Customers

customers = dados['customers'].copy()

#Remover duplicatas

customers = customers.drop_duplicates(subset=['customer_id'])

#Padronização de texto

for col in ['customers_city', 'customers_state']:
    if col in customers.columns:
        customers[col] = customers[col].str.upper().set.strip()

#Salvar dataset

salvar_dataset(customers,'customers_clean.csv')
dados['customers'] = customers

In [ ]:
#Limpeza de dados da tabela Products

products = dados['products'].copy()

#Preenchendo valores nulos com 'unknown'

products['product_category_name'].fillna('unknown', inplace=True)

#Preencher dimensões nulas com mediana
for col in ['products_weight_g', 'products_length_cm', 'products_height_cm', 'products_width_cm']:
    if col in products.columns:
        products[col].fillna(products[col].median(), inplace=True)

#Padronizar texto
products['product_category_name'] = products['product_category_name'].str.lower().str.strip()

#Remover duplicatas
products = products.drop_duplicates(subset=['product_id'])

#Remover produtos sem id

products = products[products['product_id'].notna()]

salvar_dataset(products,'products_clean.csv')